# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ak470107/ML-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Tables:** `fact_content_daily_performance` (primary — client x content x day, partitioned by `month=YYYY-MM`), joined against `dim_content` for content metadata and `dim_clients` for per-client history coverage.

**One row, raw grain:** in `fact_content_daily_performance`, one row = one content item, for one client, on one calendar day (`report_date x client_hash_id x content_hash_id`). That is finer than my lane's analysis unit — Lane 2 (Refresh / Content Opportunity Scoring) reasons about *content items*, not content-days. So every feature I build below rolls the daily grain up to **one row = one content item**, by aggregating its daily rows over a chosen window. I verify the raw grain holds (no duplicate content-day rows) before I trust any rollup of it.

**Time window:** I iterate on the mid-panel partition `month=2026-03` only, per the warning that `fact_content_daily_performance_sample` is the sealed final month (June 2026) and must never be used to develop label logic. Within March I treat **2026-03-31 as the decision moment** — every feature below is built only from rows dated on or before that day, entirely inside the March partition, so nothing from later months leaks in.

In [2]:
%pip -q install duckdb

import os, getpass
import pandas as pd
import numpy as np

# Token order: env var -> Colab Secret -> prompt (last resort).
# Never paste the token directly into a cell -- this repo is public.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':  f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':  f"read_parquet('{REL}/dim_content.parquet')",
    'fact_march':   f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    'fact_sample':  f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",  # sealed final month -- mechanics only, never for label logic
}

# COUNT(*) + MIN/MAX(date) touch Parquet metadata, not data -- near-free, and the first
# thing to run in any session to confirm we're pointed at the right partition.
check = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT client_hash_id)  AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {TABLES['fact_march']}
""").df()
check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_clients,n_content,min_date,max_date
0,9841378,55,331437,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Fields | Why |
|---|---|---|
| **Context** (grouping/joining only) | `client_hash_id`, `content_hash_id`, `report_date` | Pseudonymous IDs and the date column place a row — they're never signal for a model to learn from |
| **Feature** (knowable at the decision moment) | `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, days-with-impressions coverage, GA4 engagement rate — all rolled up over March **up to** the decision moment | Everything a content manager could already see in the dashboard by 2026-03-31 |
| **Label / proxy** (what I predict — never a feature) | An `is_declining` flag: impressions fell 20%+ from an earlier sub-window to a later one, same logic as `trend_direction`/`trend_pct` in the starter CSV (w01–w02) | The target itself, or anything it's computed from, can never also be an input — that's the leakage trap in Part 3 |
| **Excluded** | (a) single-day raw counts (`gsc_impressions` on one specific day) — too noisy to act on, only windowed sums/averages are used; (b) `health_score` / `priority_score` / `action_type` — FlyRank's own product-decision outputs, not shipped in this dataset at all, so there's nothing to accidentally use; (c) GA4 columns on rows where `ga4_data_available` is not `TRUE` — zero-filled rows there mean "no data," not "no engagement," so they're dropped rather than treated as real zeros | Each either isn't knowable cleanly, isn't shipped, or would silently encode missingness as a real value |

**Availability flags are three-valued** (`TRUE` / `FALSE` / `NULL`), not two — so every filter on them below uses `IS TRUE` / `IS NOT TRUE`, never `= FALSE` or a bare `NOT`, which would mishandle the `NULL` rows.

In [3]:
# Verify every column I plan to touch actually exists in this partition, and see what
# else is there before I assume any GA4 column name -- a contract claim without a check
# next to it is a guess.
schema = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_march']}").df()
print(schema[['column_name', 'column_type']].to_string(index=False))

planned_context = ['client_hash_id', 'content_hash_id', 'report_date']
planned_feature_core = ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']
available_cols = set(schema['column_name'])
missing = [c for c in planned_context + planned_feature_core if c not in available_cols]
print('\nAll core planned columns present?', 'yes' if not missing else f'NO -- missing {missing}')

             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGINT
              ai_copilot      BIGINT
 

## 3. Verify it with queries (grain, counts, missing values, windows) — plus the five features and the trap

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 — grain: one raw row really is one content-day-client

In [4]:
grain_probe = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_march']}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f'Duplicate (date, client, content) combinations found: {len(grain_probe)}')
print('Grain holds -- zero rows back means the raw table really is one row per content-item-day.' if grain_probe.empty
      else 'Grain probe found duplicates -- see rows above before trusting any rollup.')
grain_probe

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (date, client, content) combinations found: 0
Grain holds -- zero rows back means the raw table really is one row per content-item-day.


,report_date,client_hash_id,content_hash_id,c


### Query 2 — my slice's row count and date span

In [5]:
counts = con.sql(f"""
    SELECT COUNT(*)                        AS n_rows,
           COUNT(DISTINCT client_hash_id)   AS n_clients,
           COUNT(DISTINCT content_hash_id)  AS n_content,
           MIN(report_date)                 AS min_date,
           MAX(report_date)                 AS max_date
    FROM {TABLES['fact_march']}
""").df()
print(f"March 2026 partition: {counts['n_rows'][0]:,} rows, "
      f"{counts['n_clients'][0]} clients, {counts['n_content'][0]:,} content items, "
      f"spanning {counts['min_date'][0]} to {counts['max_date'][0]}.")
counts

March 2026 partition: 9,841,378 rows, 55 clients, 331,437 content items, spanning 2026-03-01 00:00:00 to 2026-03-31 00:00:00.


,n_rows,n_clients,n_content,min_date,max_date
0,9841378,55,331437,2026-03-01,2026-03-31


### Query 3 — availability, filtered with `IS TRUE`

GA4 (and GSC) availability flags are three-valued. Zero-filled GA4 rows before a client's `ga4_data_start` are marked `ga4_data_available = FALSE`; millions more rows carry it as `NULL` (neither zero-filled nor flagged). Filtering with `= FALSE` or a bare `NOT` would silently keep the `NULL` rows in the "available" bucket — so I filter with `IS TRUE` and show exactly how many survive.

In [6]:
avail = con.sql(f"""
    SELECT
        COUNT(*) AS n_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE     THEN 1 ELSE 0 END) AS ga4_available_true,
        SUM(CASE WHEN ga4_data_available IS FALSE    THEN 1 ELSE 0 END) AS ga4_available_false,
        SUM(CASE WHEN ga4_data_available IS NULL     THEN 1 ELSE 0 END) AS ga4_available_null
    FROM {TABLES['fact_march']}
""").df()

total = avail['n_rows'][0]
survive = avail['ga4_available_true'][0]
print(f"Total March rows: {total:,}")
print(f"Survive `ga4_data_available IS TRUE`: {survive:,} ({survive/total:.1%})")
print(f"FALSE: {avail['ga4_available_false'][0]:,}   NULL: {avail['ga4_available_null'][0]:,}")
avail

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total March rows: 9,841,378
Survive `ga4_data_available IS TRUE`: 413,966.0 (4.2%)
FALSE: 6,408,671.0   NULL: 3,018,741.0


,n_rows,ga4_available_true,ga4_available_false,ga4_available_null
0,9841378,413966.0,6408671.0,3018741.0


### Five features — one row per content item, built only from March, only up to 2026-03-31

Each feature is an aggregate over daily rows dated on or before the decision moment (2026-03-31), so nothing from a later day — let alone a later month — enters it. GA4-derived features are computed only over rows where `ga4_data_available IS TRUE`.

1. `impressions_mtd` — sum of `gsc_impressions`, March 1–31. **Available when:** every daily impression count is already logged in Search Console the moment it happens; by March 31 the full month's history exists.
2. `clicks_mtd` — sum of `gsc_clicks`, March 1–31. **Available when:** same as impressions — logged daily, nothing forward-looking about it.
3. `avg_position_mtd` — mean `gsc_avg_position` across days with impressions > 0. **Available when:** rank position is observed the same day a page is shown in search results, not inferred from anything later.
4. `days_with_impressions_mtd` — count of days in March with `gsc_impressions > 0` (a consistency/coverage signal, 0–31). **Available when:** it's just a tally of already-logged days, known in full by month end.
5. `ga4_engagement_rate_mtd` — engaged sessions divided by sessions over March, GA4-available rows only. **Available when:** GA4 records engagement the same day a session happens; by the decision moment the month's sessions are already behind us.

In [7]:
# Pick the real GA4 column names from the verified schema rather than assuming them --
# same "verify, don't guess" rule as everywhere else in this notebook.
def pick(colnames, *candidates):
    for c in candidates:
        if c in colnames:
            return c
    raise ValueError(f'None of {candidates} found in schema -- check the DESCRIBE output above.')

sessions_col = pick(available_cols, 'ga4_sessions', 'sessions')
engaged_col  = pick(available_cols, 'ga4_engaged_sessions', 'engaged_sessions')

features = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions)                                             AS impressions_mtd,
        SUM(gsc_clicks)                                                  AS clicks_mtd,
        AVG(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position END)     AS avg_position_mtd,
        SUM(CASE WHEN gsc_impressions > 0 THEN 1 ELSE 0 END)             AS days_with_impressions_mtd,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN {engaged_col}  ELSE 0 END) AS ga4_engaged_mtd,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN {sessions_col} ELSE 0 END) AS ga4_sessions_mtd
    FROM {TABLES['fact_march']}
    WHERE report_date <= DATE '2026-03-31'
    GROUP BY 1, 2
    HAVING impressions_mtd >= 1
""").df()

features['ga4_engagement_rate_mtd'] = np.where(
    features['ga4_sessions_mtd'] > 0,
    features['ga4_engaged_mtd'] / features['ga4_sessions_mtd'],
    np.nan
)

print(f'{len(features):,} content items with at least one impression in March')
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

176,738 content items with at least one impression in March


,content_hash_id,client_hash_id,impressions_mtd,clicks_mtd,avg_position_mtd,days_with_impressions_mtd,ga4_engaged_mtd,ga4_sessions_mtd,ga4_engagement_rate_mtd
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,1140.0,2.0,4.394234,31.0,0.0,0.0,NaN
1,content_a7da352b73b02668,client_73cda7b4e4f265ea,4944.0,13.0,7.244844,31.0,0.0,2.0,0.0
2,content_d056587ff7faca0c,client_73cda7b4e4f265ea,2770.0,16.0,4.459107,31.0,0.0,3.0,0.0
3,content_bfd1e41c2af250c8,client_73cda7b4e4f265ea,48.0,0.0,14.753175,21.0,0.0,0.0,NaN
4,content_2662845f598544ef,client_73cda7b4e4f265ea,150.0,1.0,6.341880,30.0,0.0,1.0,0.0


### The trap — one label-derived column, on purpose

I define a small within-month demo label so I can show the leak on real data without reaching outside the March partition: `is_declining_demo` = impressions in the second half of March (16–31) fell at least 20% versus the first half (1–15). This mirrors `trend_direction`/`trend_pct` from the starter CSV — an observed swing between two windows, not an opinion.

I first score an **honest** model using only first-half features (nothing from the second half). Then I add **one column that is derived straight from the label's own second-half numerator** — `impressions_second_half` itself — retrain, and watch the score jump toward perfect. Then I delete it and confirm the honest number returns.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

halves = con.sql(f"""
    SELECT
        content_hash_id, client_hash_id,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
        SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks     ELSE 0 END) AS clk_first_half,
        AVG(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_impressions > 0 THEN gsc_avg_position END) AS pos_first_half
    FROM {TABLES['fact_march']}
    GROUP BY 1, 2
    HAVING imp_first_half >= 20
""").df()

halves['is_declining_demo'] = (halves['imp_second_half'] < 0.8 * halves['imp_first_half']).astype(int)
print('Demo base rate (share declining):', round(halves['is_declining_demo'].mean() * 100, 1), '%')

honest_cols = ['imp_first_half', 'clk_first_half', 'pos_first_half']
leak_cols   = honest_cols + ['imp_second_half']   # <- the one label-derived column, added on purpose

model_data = halves.dropna(subset=leak_cols)
X_honest = model_data[honest_cols]
X_leak   = model_data[leak_cols]
y = model_data['is_declining_demo']

X_tr_h, X_te_h, y_tr, y_te = train_test_split(X_honest, y, test_size=0.25, random_state=42, stratify=y)
X_tr_l, X_te_l, _, _       = train_test_split(X_leak,   y, test_size=0.25, random_state=42, stratify=y)

honest_model = LogisticRegression(max_iter=1000).fit(X_tr_h, y_tr)
leak_model   = LogisticRegression(max_iter=1000).fit(X_tr_l, y_tr)

honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te_h)[:, 1])
leak_auc   = roc_auc_score(y_te, leak_model.predict_proba(X_te_l)[:, 1])

print(f'Honest model  (first-half features only):                    ROC-AUC = {honest_auc:.3f}')
print(f'Leaky model   (+ imp_second_half, the label\'s own numerator): ROC-AUC = {leak_auc:.3f}')
print()
print('The leaky number jumps toward 1.0 because imp_second_half is literally what the label')
print("is thresholded from -- the model isn't finding a pattern, it's reading the answer key.")
print('Deleting that column and keeping only the honest first-half features is the real result:')
print(f'honest ROC-AUC = {honest_auc:.3f}.')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Demo base rate (share declining): 29.1 %
Honest model  (first-half features only):                    ROC-AUC = 0.601
Leaky model   (+ imp_second_half, the label's own numerator): ROC-AUC = 1.000

The leaky number jumps toward 1.0 because imp_second_half is literally what the label
is thresholded from -- the model isn't finding a pattern, it's reading the answer key.
Deleting that column and keeping only the honest first-half features is the real result:
honest ROC-AUC = 0.601.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation:** the within-month demo label above (first-half-of-March vs. second-half-of-March) is a simplified stand-in for the real trend definition used in w01–w02, which compares two full trailing 30-day windows. Two 15/16-day halves are noisier — smaller windows swing more from ordinary week-to-week variance, so `is_declining_demo`'s base rate and the honest model's AUC will both look shakier than a properly-windowed 30-vs-30 label would. I used it here only because Part 3 restricts the feature/trap work to "that same month" (`month=2026-03`); a real Lane 2 label needs an adjoining prior month (e.g. February) for the `prev30` side, which is exactly what the full capstone build does across the multi-month partition, not a single month in isolation.

Beyond that: history depth is a genuinely **unbalanced panel** (`dim_clients.gsc_data_start` differs by client — some clients simply don't have a full March), and rows before a client's `ga4_data_start` are real "no data," not zero engagement — both are already handled above (`IS TRUE` filtering) but worth restating as a hard boundary: this dataset can tell me what happened to search performance, never *why*, and it can't prove a refresh caused a recovery without a causal design this data alone doesn't provide.

In [9]:
# Supporting check for the limitation above: how many clients don't have March-long
# coverage at all, i.e. their gsc_data_start falls after March 2026 started.
coverage = con.sql(f"""
    SELECT
        COUNT(*) AS n_clients,
        SUM(CASE WHEN gsc_data_start > DATE '2026-03-01' THEN 1 ELSE 0 END) AS started_after_march_1,
        SUM(CASE WHEN gsc_data_start IS NULL THEN 1 ELSE 0 END) AS null_start
    FROM {TABLES['dim_clients']}
""").df()
print(f"Of {coverage['n_clients'][0]} total clients, "
      f"{coverage['started_after_march_1'][0]} have GSC history starting after March 1 "
      f"(no full month of March for them), and {coverage['null_start'][0]} have a NULL start date.")
coverage

Of 104 total clients, 15.0 have GSC history starting after March 1 (no full month of March for them), and 37.0 have a NULL start date.


,n_clients,started_after_march_1,null_start
0,104,15.0,37.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.